In [2]:
import pandas as pd
import pickle
from collections import defaultdict

rna_entities = pd.read_pickle("data/nc_rna_entities_to_be_processed.pkl")

# Initialize counter
db_counts = defaultdict(int)

for buckets in rna_entities['ID_buckets']:
    for db, ids in buckets.items():
        if db != 'RNAcentral':  # exclude placeholder
            db_counts[db] += len(set(ids))  # count unique IDs per entity

# Print summary
for db, count in db_counts.items():
    print(f"{db}: {count} unique IDs across entities")


miRBase: 11698 unique IDs across entities
NCBI: 39052 unique IDs across entities
Ensembl: 7496 unique IDs across entities
circBase: 258 unique IDs across entities
tRFdb: 928 unique IDs across entities
GenBank: 3 unique IDs across entities
Other: 0 unique IDs across entities


In [3]:
from collections import defaultdict

# Initialize counter
db_counts = defaultdict(int)

for buckets in rna_entities['ID_buckets']:
    for db, ids in buckets.items():
        if db != 'RNAcentral' and ids:  # exclude placeholder and empty lists
            db_counts[db] += 1  # count the row/entity, not unique IDs

# Print summary
for db, count in db_counts.items():
    print(f"{db}: {count} rows containing IDs")


Ensembl: 7496 rows containing IDs
NCBI: 39052 rows containing IDs
miRBase: 11672 rows containing IDs
circBase: 257 rows containing IDs
GenBank: 3 rows containing IDs
tRFdb: 928 rows containing IDs


In [4]:
'''
Assumption: all annotated isoforms (transcripts) of a gene are considered potential interactors 
when RNAInter reports a gene-level interaction. We keep track of isoforms-per-interaction (A list of sequences)
'''

'\nAssumption: all annotated isoforms (transcripts) of a gene are considered potential interactors \nwhen RNAInter reports a gene-level interaction. We keep track of isoforms-per-interaction (A list of sequences)\n'

<h3>Ensembl IDs to Ensemble transcripts to RNAcentral IDs</h3>

In [5]:
# Construct DUCK-DB based on RNAcentral BEFORE this step!!
import sys
sys.path.append("rnacentral-etl")

from rnacentral import QueryRnaCentral

qrc = QueryRnaCentral(base_dir="rnacentral_data/current_release", db_path="rnacentral-etl/rnacentral.duckdb")

In [6]:
'''
Extract all RNAcentral IDS of the ensemble transcripts related to ensemble GeneIDs that satisfy the conditions (Tax ID and rna-type)
Extract from row.get("ID_buckets", {}).get("Ensembl", [])
and place in RNAcentral a list for each transcript, so that we have a list of lists. row.get("ID_buckets", {}).get("RNAcentral", [])
We are not sure if all ENSEMBL IDs we have are gene ids, so first try with gene, if nothing try with transcript ID.
'''

# Usefoul code: 
taxid= "10090"
res = qrc.ensembl_gene_to_rnacentral_transcripts("ENSMUSG00000065862", taxid) #for ENS. gene ID
# qrc.ensembl_to_rnacentral('ENSMUST00000083928', taxid) # ENSMUSG00000065862 for ENS transcript ID
'''
Ensure 'rna_type' returned by query rna central is the same rna_type of the row.
First look if the rnacentral type is in the standard categories.
'''

res

,RNAcentral_ID,rna_type,taxid,external_db
0,URS0000639553,snoRNA,10090,ENSEMBL


In [11]:
STANDARD_RNA_TYPES = {
    "tRNA", "miRNA", "rRNA", "pseudo", "lncRNA", "piRNA",
    "sRNA", "lincRNA", "ncRNA", "ribozyme", "snoRNA",
    "scaRNA", "snRNA", "scRNA", "sncRNA",
    "shRNA", "circRNA", "tRF", "eRNA"
}

ACCEPTABLE_TYPE_MAP = {
    "SRP_RNA": ["ncRNA"],
    "RNase_MRP_RNA": ["ncRNA", "ribozyme"],
    "pre_miRNA": ["miRNA"],          # precursor -> mature
    "scaRNA": ["snoRNA", "ncRNA"],
    "snRNA": ["ncRNA"],               # small nuclear RNA -> generic ncRNA
    "misc_RNA": ["ncRNA", "rRNA", "snRNA", "snoRNA"],
    "ncRNA": ["snRNA"],  
    #"": [""],
    #"": [""]
}

def get_rnacentral_ids(row):
    """
    Populate row['ID_buckets']['RNAcentral'] with:
    a list of lists (one list per Ensembl ID).
    """
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    ensembl_ids = id_buckets.get("Ensembl", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")

    if not ensembl_ids or taxid is None:
        return row

    rnacentral_per_ensembl = []
    
    for ensembl_id in ensembl_ids:
        rnacentral_ids = set()

        try:
            # 1) try Ensembl gene → RNAcentral transcripts
            df = qrc.ensembl_gene_to_rnacentral_transcripts(
                ensembl_id, taxid
            )

            # 2) fallback: Ensembl transcript → RNAcentral
            if df is None or df.empty:
                df = qrc.ensembl_to_rnacentral(
                    ensembl_id, taxid
                )
                if df is not None and not df.empty:
                    print('Transcript detected')
                    print(df)

            if df is None or df.empty:
                continue

            for _, r in df.iterrows():
                rc_id = r.get("RNAcentral_ID")
                rc_type = r.get("rna_type")

                if pd.isna(rc_id) or pd.isna(rc_type):
                    print('rnacentral data not valid')
                    continue

                #if rc_type not in STANDARD_RNA_TYPES:
                #    print(f"[UNSTANDARDISED RNA TYPE] {rc_type}, row type in this case: {row_rna_type}")
                
                # Check if the rc_type is OK for the row type
                if rc_type == row_rna_type:
                    rnacentral_ids.add(rc_id)
                elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                    # defensible bypass
                    rnacentral_ids.add(rc_id)
                else:
                    # truly mismatched type — log for review
                    print(f"[NO TYPE MATCH] {rc_type}, row type in this case: {row_rna_type}")

        except Exception:
            continue

        if rnacentral_ids:
            rnacentral_per_ensembl.append(list(rnacentral_ids))

    id_buckets["RNAcentral"] = rnacentral_per_ensembl if rnacentral_per_ensembl else []
    row["ID_buckets"] = id_buckets

    return row


rna_entities_trial = rna_entities.apply(
    get_rnacentral_ids, axis=1
)


[NO TYPE MATCH] lncRNA, row type in this case: pseudo
[NO TYPE MATCH] lncRNA, row type in this case: pseudo
[NO TYPE MATCH] sRNA, row type in this case: miRNA
[NO TYPE MATCH] sRNA, row type in this case: miRNA
[NO TYPE MATCH] sRNA, row type in this case: miRNA


In [12]:
from collections import defaultdict
import numpy as np

def compute_stats(df, target_db="RNAcentral"):
    """
    Compute row-level and ID-level statistics for ID_buckets.
    Row = entity (primary unit).
    """

    # -------------------------
    # Row-level statistics
    # -------------------------
    db_rows_with_ids = defaultdict(int)
    db_ids_per_row = defaultdict(list)

    for _, row in df.iterrows():
        buckets = row.get("ID_buckets", {})
        for db, ids in buckets.items():
            if not ids:
                continue

            # count this row once per DB
            db_rows_with_ids[db] += 1

            # number of IDs in this row for this DB
            n_ids = 0
            for x in ids:
                n_ids += len(x) if isinstance(x, list) else 1

            db_ids_per_row[db].append(n_ids)

    print("=== Row-level (entity-level) statistics ===")
    for db, counts in db_ids_per_row.items():
        arr = np.array(counts)
        print(
            f"{db}: rows={db_rows_with_ids[db]} | "
            f"IDs/row: max={arr.max()}, mean={arr.mean():.2f}, median={np.median(arr)}"
        )

    # -------------------------
    # Row-level bins (IDs per row)
    # -------------------------
    print("\n=== IDs per row (binned) ===")
    for db, counts in db_ids_per_row.items():
        count_1 = np.sum(np.array(counts) == 1)
        count_2 = np.sum(np.array(counts) == 2)
        count_3_10 = np.sum((np.array(counts) >= 3) & (np.array(counts) <= 10))
        count_gt10 = np.sum(np.array(counts) > 10)

        print(
            f"{db}: 1={count_1}, 2={count_2}, "
            f"3–10={count_3_10}, >10={count_gt10}"
        )

    # -------------------------
    # ID-level statistics (secondary)
    # -------------------------
    db_unique_ids = defaultdict(set)

    for _, row in df.iterrows():
        buckets = row.get("ID_buckets", {})
        for db, ids in buckets.items():
            for x in ids:
                if isinstance(x, list):
                    db_unique_ids[db].update(x)
                else:
                    db_unique_ids[db].add(x)

    print("\n=== ID-level (secondary) statistics ===")
    for db, ids in db_unique_ids.items():
        print(f"{db}: {len(ids)} unique IDs")

    # -------------------------
    # Sequence availability (target DB)
    # -------------------------
    if target_db:
        has_seq = sum(
            bool(row.get("ID_buckets", {}).get(target_db))
            for _, row in df.iterrows()
        )
        no_seq = len(df) - has_seq

        print("\n=== Sequence availability ===")
        print(f"{target_db}: rows with IDs = {has_seq}")
        print(f"{target_db}: rows without IDs = {no_seq}")
        print(f"{target_db}: fraction retained = {has_seq / len(df):.3f}")

    # -------------------------
    # Sequence availability (target DB OR fallback sequence)
    # -------------------------
    print('new sequence avibility counting sequences too')
    if target_db:
        has_retained = 0
    
        for _, row in df.iterrows():
            id_buckets = row.get("ID_buckets", {})
            has_ids = bool(id_buckets.get(target_db))
    
            seqs = row.get("sequences")
            has_sequences = isinstance(seqs, list) and len(seqs) >= 1
    
            if has_ids or has_sequences:
                has_retained += 1
    
        not_retained = len(df) - has_retained
    
        print("\n=== Sequence availability ===")
        print(f"{target_db} IDs OR fallback sequence present = {has_retained}")
        print(f"Rows without sequence information = {not_retained}")
        print(f"Fraction retained = {has_retained / len(df):.3f}")
    


In [13]:
compute_stats(rna_entities_trial)

=== Row-level (entity-level) statistics ===
Ensembl: rows=7496 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=4285 | IDs/row: max=249, mean=4.80, median=1.0
NCBI: rows=39052 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=11672 | IDs/row: max=3, mean=1.00, median=1.0
circBase: rows=257 | IDs/row: max=2, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
tRFdb: rows=928 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=7496, 2=0, 3–10=0, >10=0
RNAcentral: 1=2640, 2=395, 3–10=810, >10=440
NCBI: 1=39052, 2=0, 3–10=0, >10=0
miRBase: 1=11650, 2=18, 3–10=4, >10=0
circBase: 1=256, 2=1, 3–10=0, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
tRFdb: 1=928, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 7496 unique IDs
RNAcentral: 20504 unique IDs
NCBI: 38441 unique IDs
miRBase: 11332 unique IDs
circBase: 208 unique IDs
GenBank: 3 unique IDs
tRFdb: 463 unique IDs

=== Sequence availability ===
RNAcentral: rows

In [14]:
rna_entities_trial.to_pickle("data/tmp/rna_entities_trial.pkl")

In [15]:
# Construct DUCK-DB based on RNAcentral BEFORE this step!!
import sys
import pandas as pd
sys.path.append("rnacentral-etl")

from rnacentral import QueryRnaCentral

qrc = QueryRnaCentral(base_dir="rnacentral_data/current_release", db_path="rnacentral-etl/rnacentral.duckdb")

rna_entities_trial = pd.read_pickle("data/tmp/rna_entities_trial.pkl")

In [16]:
# -----------------------------
# Load NCBI → Ensembl mapping
# -----------------------------
ncbi_to_ensemble = pd.read_csv(
    "data/gene2ensembl",
    sep="\t",
    dtype=str,
    comment="#",
    header=None,
    low_memory=False,
)

ncbi_to_ensemble.columns = [
    "taxid",
    "geneid",
    "ensembl_gene",
    "refseq_rna_version",
    "ensembl_transcript_id",
    "refseq_protein_version",
    "ensembl_protein_id",
]

# enforce string typing
ncbi_to_ensemble["taxid"] = ncbi_to_ensemble["taxid"].astype(str)
ncbi_to_ensemble["geneid"] = ncbi_to_ensemble["geneid"].astype(str)

# collect all GeneIDs used in your dataset
all_ncbi_ids = set()

for buckets in rna_entities_trial["ID_buckets"]:
    all_ncbi_ids.update(map(str, buckets.get("NCBI", [])))

print(f"{len(all_ncbi_ids)} unique NCBI GeneIDs")

ncbi_to_ensemble_small = ncbi_to_ensemble[
    (ncbi_to_ensemble["ensembl_gene"] != "-") &
    (ncbi_to_ensemble["geneid"].isin(all_ncbi_ids))
]

ncbi2ens_gene = (
    ncbi_to_ensemble_small
    .groupby(["taxid", "geneid"])["ensembl_gene"]
    .unique()
    .to_dict()
)
print(len(ncbi2ens_gene))

import pickle
from functools import lru_cache

# key = rc_type from RNAcentral, value = list of row types it can match

38441 unique NCBI GeneIDs
15632


<h3>NCBI IDs to refSeq IDs to RNAcentral IDs</h3>

In [17]:
from collections import defaultdict
from functools import lru_cache

gene2refseq_index = defaultdict(list)

with open("data/gene2refseq", "r") as f:   # <-- FIX: no gzip
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")

        taxid = cols[0]
        geneid = cols[1]
        rna_acc = cols[3]   # RNA_nucleotide_accession
        status = cols[2]

        if status == "SUPPRESSED":
            continue

        if rna_acc == "-":
            continue

        if rna_acc.startswith(("NR_", "XR_")):
            rna_type = "ncRNA"
        #elif rna_acc.startswith(("NM_", "XM_")):
        #    continue
            #rna_type = "mRNA"
        else:
            continue

        gene2refseq_index[(taxid, geneid)].append({
            "refseq_id": rna_acc,
            "rna_type": rna_type,
            "status": status
        })



In [18]:
@lru_cache(maxsize=100_000)
def ncbi_gene_to_refseq_transcripts(ncbi_gene_id, taxid):
    key = (str(ncbi_gene_id), str(taxid))  # cast to str
    return gene2refseq_index.get(key, [])

print(ncbi_gene_to_refseq_transcripts(2711, 102577949))

refseq_id = "NR_036832"  # example RefSeq
taxid = "58123"
print(f"\nRefSeq to RNAcentral (RefSeq={refseq_id}, taxid={taxid}):")
df_ref = qrc.refseq_to_rnacentral(refseq_id, taxid)
print(df_ref)

[{'refseq_id': 'XR_003065960.2', 'rna_type': 'ncRNA', 'status': 'MODEL'}, {'refseq_id': 'XR_371036.4', 'rna_type': 'ncRNA', 'status': 'MODEL'}]

RefSeq to RNAcentral (RefSeq=NR_036832, taxid=58123):
   RNAcentral_ID rna_type
0  URS0000000036     rRNA


In [19]:
def extract_refseq_transcript_type(refseq_id: str):
    try:
        handle = Entrez.efetch(
            db="nuccore",
            id=refseq_id,
            rettype="gb",
            retmode="text"
        )
        record = SeqIO.read(handle, "genbank")
        handle.close()

        for feature in record.features:
            if feature.type == "ncRNA":
                qualifiers = feature.qualifiers
                
                if "ncRNA_class" in qualifiers:
                    return qualifiers["ncRNA_class"][0]

                if "product" in qualifiers:
                    return qualifiers["product"][0]

        return None

    except Exception as e:
        print("Transcript fetch failed:", refseq_id, e)
        return None


In [ ]:
from Bio import Entrez, SeqIO
from io import StringIO

Entrez.email = "marco.nicolini@unimi.it" 

def fetch_refseq_sequence(refseq_id: str) -> str:
    handle = Entrez.efetch(
        db="nuccore",
        id=refseq_id,
        rettype="fasta",
        retmode="text"
    )
    fasta_text = handle.read()
    handle.close()
    
    record = SeqIO.read(StringIO(fasta_text), "fasta")
    return str(record.seq)
    
def normalize_sequence(seq: str) -> str:
    seq = seq.upper()
    return seq

errors_standard_types = []
errors_no_match_types = []

def get_rnacentral_from_ncbi_ids_refseq(row):
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    ncbi_ids = id_buckets.get("NCBI", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")

    if not ncbi_ids or not taxid:
        return row

    # do not overwrite existing RNAcentral mappings
    if id_buckets.get("RNAcentral"):
        return row

    rnacentral_per_ncbi = []
    sequences = []

    for ncbi_id in ncbi_ids:
        rnacentral_ids = set()
        try:
            refseq_records = ncbi_gene_to_refseq_transcripts(
                taxid, ncbi_id
            )
            if not refseq_records:
                continue

            for rec in refseq_records:
                refseq_id = rec.get("refseq_id")
                refseq_id = refseq_id.split(".")[0]
                status = rec.get("status")
                

                if not refseq_id:
                    continue

                df_rnacentral = qrc.refseq_to_rnacentral(str(refseq_id), str(taxid))

                if df_rnacentral is None or df_rnacentral.empty:
                    if not rnacentral_ids: # if we have good RNAcentral matches we can skip
                        try:
                            # download sequence
                            seq = fetch_refseq_sequence(str(refseq_id))
                            seq = normalize_sequence(seq)
                            seq_Us = seq.replace("T", "U")
                            
                        except Exception as e:
                            print("Sequence fetch failed:", refseq_id, e)
                            continue
                        
                        df_rnacentral = qrc.sequence_to_rnacentral(seq, str(taxid))
                        
                        if df_rnacentral is None or df_rnacentral.empty:
                            df_rnacentral = qrc.sequence_to_rnacentral(seq_Us, str(taxid))
                        if df_rnacentral is None or df_rnacentral.empty:
                            # we check for REFSEQ TYPE, no RNAcentral Mapping
                            refseq_type = extract_refseq_transcript_type(str(refseq_id))
                            if refseq_type and refseq_type.lower() == row_rna_type.lower():
                                sequences.append(seq_Us)
                            # else: no check
                            elif refseq_type is None:
                                # keep only if RefSeq accession indicates non-coding RNA
                                if refseq_id.startswith(("NR_", "XR_")):
                                    sequences.append(seq_Us)
                                    row["typecheck"] = False
                            continue 
                        
                # For loop for Rnacentral items:
                for _, row_rc in df_rnacentral.iterrows():
                    rnacentral_id = row_rc.get('RNAcentral_ID')
                    rc_type = row_rc.get('rna_type')

                    if rc_type not in STANDARD_RNA_TYPES:
                        errors_standard_types.append(
                            f"[UNSTANDARDISED RNA TYPE] {rc_type} (refseq_id:{refseq_id})"
                        )
                        
                    # Check if the rc_type is OK for the row type
                    if rc_type == row_rna_type:
                        rnacentral_ids.add(rnacentral_id)
                    elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                        # defensible bypass
                        rnacentral_ids.add(rnacentral_id)
                    else:
                        # truly mismatched type — log for review
                        errors_no_match_types.append(f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}")

        except Exception as e:
            print("Error:", e)
            continue

        if rnacentral_ids:
            rnacentral_per_ncbi.append(list(rnacentral_ids))

    id_buckets["RNAcentral"] = rnacentral_per_ncbi
    row["ID_buckets"] = id_buckets
    if not rnacentral_per_ncbi and sequences:
        row["sequences"] = sequences
    return row

from tqdm.auto import tqdm
tqdm.pandas()

rna_entities_trial_stepTwo = rna_entities_trial.progress_apply(
    get_rnacentral_from_ncbi_ids_refseq,
    axis=1
)

rna_entities_trial_stepTwo.to_pickle("data/tmp/rna_entities_trial_stepTwo.pkl")

compute_stats(rna_entities_trial_stepTwo)


In [21]:
rna_entities_trial_stepTwo.to_pickle("data/tmp/rna_entities_trial_stepTwo.pkl")

compute_stats(rna_entities_trial_stepTwo)

=== Row-level (entity-level) statistics ===
Ensembl: rows=7496 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=25307 | IDs/row: max=249, mean=1.77, median=1.0
NCBI: rows=39052 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=11672 | IDs/row: max=3, mean=1.00, median=1.0
circBase: rows=257 | IDs/row: max=2, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
tRFdb: rows=928 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=7496, 2=0, 3–10=0, >10=0
RNAcentral: 1=22376, 2=1091, 3–10=1363, >10=477
NCBI: 1=39052, 2=0, 3–10=0, >10=0
miRBase: 1=11650, 2=18, 3–10=4, >10=0
circBase: 1=256, 2=1, 3–10=0, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
tRFdb: 1=928, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 7496 unique IDs
RNAcentral: 44423 unique IDs
NCBI: 38441 unique IDs
miRBase: 11332 unique IDs
circBase: 208 unique IDs
GenBank: 3 unique IDs
tRFdb: 463 unique IDs

=== Sequence availability ===
RNAcentral: 

<h3>NCBI IDs to Ensembl IDs to Ensemble transcripts to RNAcentral IDs</h3>

In [22]:
@lru_cache(maxsize=100_000)
def cached_ensembl_gene_to_rnacentral(ensembl_gene, taxid):
    return qrc.ensembl_gene_to_rnacentral_transcripts(ensembl_gene, taxid)

# -----------------------------
# Main function
# -----------------------------
errors_standard_types = []
errors_no_match_types = []
def get_rnacentral_from_ncbi_ids(row):
    """
    Starting from NCBI GeneIDs:
    - map to Ensembl transcripts via gene2ensembl
    - query RNAcentral at transcript level
    - enforce RNA type consistency
    - store list-of-lists in ID_buckets['RNAcentral']
      (one list per NCBI GeneID)
    - append to existing RNAcentral IDs if present
    """
    row = row.copy()   # critical
    id_buckets = row.get("ID_buckets", {})
    ncbi_ids = id_buckets.get("NCBI", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")  

    if not ncbi_ids or taxid is None:
        return row

    # skip existing RNAcentral entries, if any
    existing = id_buckets.get("RNAcentral", [])
    if existing:
        #print(existing)
        #print(id_buckets)
        return row
    rnacentral_per_ncbi = []

    for ncbi_id in ncbi_ids:
        rnacentral_ids = set()

        try:
            # NCBI GeneID → Ensembl Gene IDs
            ens_genes = ncbi2ens_gene.get((taxid, str(ncbi_id)), [])
            
            for ensembl_gene in ens_genes:
                #df = qrc.ensembl_gene_to_rnacentral_transcripts(
                #    ensembl_gene, taxid
                #)
                df = cached_ensembl_gene_to_rnacentral(ensembl_gene, taxid)

                if df is None or df.empty:
                    continue

                for _, r in df.iterrows():
                    rc_id = r.get("RNAcentral_ID") or r.get("rnacentral_id")
                    rc_type = r.get("rna_type")
                    
                    if pd.isna(rc_id) or pd.isna(rc_type):
                        continue

                    # Normalize unstandardised RNA types if needed
                    if rc_type not in STANDARD_RNA_TYPES:
                        errors_standard_types.append(f"[UNSTANDARDISED RNA TYPE] {rc_type} for row type {row_rna_type}")
                    
                    # Check if the rc_type is OK for the row type
                    if rc_type == row_rna_type:
                        rnacentral_ids.add(rc_id)
                    elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                        # defensible bypass
                        rnacentral_ids.add(rc_id)
                    else:
                        # truly mismatched type — log for review
                        errors_no_match_types.append(f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}")
        except Exception:
            continue

        # Only append non-empty lists
        if rnacentral_ids:
            rnacentral_per_ncbi.append(list(rnacentral_ids))

    # If nothing found and no previous entries, make it empty
    id_buckets["RNAcentral"] = rnacentral_per_ncbi if rnacentral_per_ncbi else []
    row["ID_buckets"] = id_buckets

    return row

# -----------------------------
# Apply to dataframe
# -----------------------------

from tqdm import tqdm

tqdm.pandas(desc="Querying RNAcentral from NCBI IDs")

rna_entities_trial_stepThree = rna_entities_trial_stepTwo.progress_apply(
    get_rnacentral_from_ncbi_ids,
    axis=1
)

rna_entities_trial_stepThree.to_pickle("data/tmp/rna_entities_trial_stepThree.pkl")

compute_stats(rna_entities_trial_stepThree)

Querying RNAcentral from NCBI IDs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 76961/76961 [05:29<00:00, 233.64it/s]


=== Row-level (entity-level) statistics ===
Ensembl: rows=7496 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=25546 | IDs/row: max=249, mean=1.89, median=1.0
NCBI: rows=39052 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=11672 | IDs/row: max=3, mean=1.00, median=1.0
circBase: rows=257 | IDs/row: max=2, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
tRFdb: rows=928 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=7496, 2=0, 3–10=0, >10=0
RNAcentral: 1=22448, 2=1109, 3–10=1431, >10=558
NCBI: 1=39052, 2=0, 3–10=0, >10=0
miRBase: 1=11650, 2=18, 3–10=4, >10=0
circBase: 1=256, 2=1, 3–10=0, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
tRFdb: 1=928, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 7496 unique IDs
RNAcentral: 47652 unique IDs
NCBI: 38441 unique IDs
miRBase: 11332 unique IDs
circBase: 208 unique IDs
GenBank: 3 unique IDs
tRFdb: 463 unique IDs

=== Sequence availability ===
RNAcentral: 

<h3>miRbase IDs to mature miRbase IDs to RNAcentral IDs</h3>

In [ ]:
#import pandas as pd
#rna_entities_trial_stepThree = pd.read_pickle("rna_entities_trial_stepThree.pkl")

In [23]:
qrc.mirbase_to_rnacentral('MIMAT0000434', '9606')

,RNAcentral_ID,rna_type,taxid
0,URS00002620A7,miRNA,9606


In [24]:
from Bio import SeqIO
from collections import defaultdict

# precursor_id -> list of mature IDs (MIMAT…)
precursor_to_mature_ids = defaultdict(list)

# mature_id -> sequence
mature_to_sequence = defaultdict(list)

# precursor_id -> fallback sequences
precursor_to_sequence = defaultdict(list)

for record in SeqIO.parse("data/miRNA.dat", "embl"):
    precursor_id = record.id  # MI000...

    for feature in record.features:
        if feature.type not in ("miRNA", "mature_miRNA"):
            continue

        seq = feature.location.extract(record).seq.upper().replace("T", "U")
        quals = feature.qualifiers

        mat_acc = quals.get("accession", [None])[0]

        if mat_acc:  # mature miRNA
            mature_to_sequence[mat_acc].append(str(seq))
            precursor_to_mature_ids[precursor_id].append(mat_acc)
        else:
            # precursor-only sequence (fallback)
            precursor_to_sequence[precursor_id].append(str(seq))


In [25]:
def get_mirna_sequences(row):
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    taxid = str(row.get("Species"))

    mirbase_ids = id_buckets.get("miRBase", [])
    rnacentral_per_mirbase = []
    seqs = set()

    if not mirbase_ids:
        return row

    rnacentral_ids = set()

    for mid in mirbase_ids:
        mature_ids = []

        if mid.startswith("MIMAT"):
            mature_ids = [mid]
        else:
            # precursor → mature IDs
            mature_ids = precursor_to_mature_ids.get(mid, [])

            # fallback: no mature annotated
            if not mature_ids:
                seqs.update(precursor_to_sequence.get(mid, []))
                continue

        for mature_id in mature_ids:
            # sequences
            seqs.update(mature_to_sequence.get(mature_id, []))

            # RNAcentral mapping
            df_rc = qrc.mirbase_to_rnacentral(mature_id, taxid)
            if df_rc is not None and not df_rc.empty:
                rnacentral_ids.update(df_rc["RNAcentral_ID"].tolist())

    if rnacentral_ids:
        rnacentral_per_mirbase.append(list(rnacentral_ids))

    row["sequences"] = list(seqs)
    id_buckets["RNAcentral"] = rnacentral_per_mirbase
    row["ID_buckets"] = id_buckets

    return row


In [26]:
rna_entities_trial_stepFour = rna_entities_trial_stepThree.apply(
    get_mirna_sequences, axis=1
)

In [27]:
compute_stats(rna_entities_trial_stepFour) # new

=== Row-level (entity-level) statistics ===
Ensembl: rows=7496 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=36870 | IDs/row: max=249, mean=1.65, median=1.0
NCBI: rows=39052 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=11672 | IDs/row: max=3, mean=1.00, median=1.0
circBase: rows=257 | IDs/row: max=2, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
tRFdb: rows=928 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=7496, 2=0, 3–10=0, >10=0
RNAcentral: 1=32484, 2=2384, 3–10=1444, >10=558
NCBI: 1=39052, 2=0, 3–10=0, >10=0
miRBase: 1=11650, 2=18, 3–10=4, >10=0
circBase: 1=256, 2=1, 3–10=0, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
tRFdb: 1=928, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 7496 unique IDs
RNAcentral: 55425 unique IDs
NCBI: 38441 unique IDs
miRBase: 11332 unique IDs
circBase: 208 unique IDs
GenBank: 3 unique IDs
tRFdb: 463 unique IDs

=== Sequence availability ===
RNAcentral: 

In [28]:
rna_entities_trial_stepFour.to_pickle("data/tmp/rna_entities_trial_stepFour.pkl")

<h3>Final step for sequence conversion: Symbol from RNAInter</h3>

In [29]:
import pandas as pd
#rna_entities_trial_stepFour = pd.read_pickle("rna_entities_trial_stepFour.pkl")

# we start from rna_entities_trial_stepFour
# 1. for each row we search if RNAcentral has a corresponding external ID, with related taxID and ncRNA type.
# 2. if the corresponding ID exists, and RNAcentral id bucket is empty, add the id/s to the bucket. Nice!
# 3. if the corresponding ID exist, and it is only one, and RNAcentral id bucket is not empty: substitute RNAcentral id bucket with the new RNAcentral ID
# 4. if the corresponding ID exist, and it is > 1, and RNAcentral id bucket is not empty: add new RNAcentral IDs, if not present already.

# Keep track of point 1, 2, 3 and 4 numbers (How many we added.)


qrc.external_to_rnacentral('Gm24029', '10090')
 

,RNAcentral_ID,rna_type,external_db,external_id
0,URS0000639553,snoRNA,MGI,MGI:5453806


In [30]:
import pandas as pd
import numpy as np

stats = {
    "matched_external_id": 0,
    "bucket_filled_empty": 0,
    "bucket_replaced_single": 0,
    "bucket_extended_multiple": 0,
    "bucket_extended_multiple_notranscript": 0
}

errors_standard_types = []
errors_no_match_types = []

def flatten_bucket(bucket):
    flat = []
    for x in bucket:
        if isinstance(x, list):
            flat.extend(x)
        else:
            flat.append(x)
    return flat

for idx, row in rna_entities_trial_stepFour.iterrows():

    ext_id = row["Symbol"]
    taxid = str(row["Species"])
    row_rna_type = row["Category"]

    id_buckets = row["ID_buckets"]
    bucket = flatten_bucket(id_buckets.get("RNAcentral", []))

    df_match = qrc.external_to_rnacentral(ext_id, taxid)

    if df_match.empty:
        continue

    stats["matched_external_id"] += 1

    accepted_rows = []

    for _, r in df_match.iterrows():
        rc_id = r["RNAcentral_ID"]
        rc_type = r["rna_type"]
        rc_db = r.get("external_db")

        if pd.isna(rc_id) or pd.isna(rc_type):
            continue

        if rc_type not in STANDARD_RNA_TYPES:
            errors_standard_types.append(
                f"[UNSTANDARDISED RNA TYPE] {rc_type} for row type {row_rna_type}"
            )

        if rc_type == row_rna_type:
            accepted_rows.append(r)
        elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
            accepted_rows.append(r)
        else:
            errors_no_match_types.append(
                f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}"
            )

    if not accepted_rows:
        continue

    df_resulting = pd.DataFrame(accepted_rows)

    # Prefer RefSeq-backed mappings if ambiguity
    if len(df_resulting) > 1 and "external_db" in df_resulting.columns:
        df_refseq = df_resulting[df_resulting["external_db"] == "REFSEQ"]
        if not df_refseq.empty:
            df_resulting = df_refseq

    new_ids = df_resulting["RNAcentral_ID"].unique().tolist()

    # ---- bucket logic ----
    bucket_set = set(bucket)
    new_set = set(new_ids)
    
    if len(bucket_set) == 0:
        # nothing existed → initialize
        bucket = sorted(new_set)
        stats["bucket_filled_empty"] += 1
    
    elif bucket_set.intersection(new_set):
        # at least one ID matches → treat new mapping as a refinement
        # (replace with the coherent new set)
        bucket = sorted(new_set)
        stats["bucket_replaced_single"] += 1
    
    else:
        # completely new evidence → extend, do not overwrite
        bucket = sorted(bucket_set.union(new_set))
    
        if len(new_set) == 1:
            stats["bucket_extended_multiple_notranscript"] += 1
        else:
            stats["bucket_extended_multiple"] += 1
    
    id_buckets["RNAcentral"] = bucket
    rna_entities_trial_stepFour.at[idx, "ID_buckets"] = id_buckets


In [31]:
stats # new

{'matched_external_id': 34516,
 'bucket_filled_empty': 1241,
 'bucket_replaced_single': 30292,
 'bucket_extended_multiple': 116,
 'bucket_extended_multiple_notranscript': 1820}

In [32]:
compute_stats(rna_entities_trial_stepFour) # new

=== Row-level (entity-level) statistics ===
Ensembl: rows=7496 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=38111 | IDs/row: max=454, mean=1.82, median=1.0
NCBI: rows=39052 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=11672 | IDs/row: max=3, mean=1.00, median=1.0
circBase: rows=257 | IDs/row: max=2, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
tRFdb: rows=928 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=7496, 2=0, 3–10=0, >10=0
RNAcentral: 1=32483, 2=2190, 3–10=2768, >10=670
NCBI: 1=39052, 2=0, 3–10=0, >10=0
miRBase: 1=11650, 2=18, 3–10=4, >10=0
circBase: 1=256, 2=1, 3–10=0, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
tRFdb: 1=928, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 7496 unique IDs
RNAcentral: 62235 unique IDs
NCBI: 38441 unique IDs
miRBase: 11332 unique IDs
circBase: 208 unique IDs
GenBank: 3 unique IDs
tRFdb: 463 unique IDs

=== Sequence availability ===
RNAcentral: 

In [33]:
rna_entities_trial_stepFour.to_pickle("data/tmp/rna_entities_trial_stepFour.pkl")

<h2>RNAcentral sequence retrieval, and GO terms retrieval</h2>

In [34]:
rna_entities_trial_stepFour = pd.read_pickle("data/tmp/rna_entities_trial_stepFour.pkl")

In [35]:
# input dataframe: rna_entities_trial_stepFour
# 1. retrieve RNAcentral sequences: rnacentral_to_sequence -> list in 'sequences' column
# 2. retrieve GO terms and rfam_identifiers with rnacentral_metadata -> add 'GO' column, NaN if no GO, otherwhise list of correlated GO terms
# 3. RfamId, same as GO terms


In [36]:
from collections import defaultdict
import pandas as pd
import re
from tqdm import tqdm

# ----------------------------
# Helper functions
# ----------------------------

def normalize_rnacentral_bucket(bucket):
    """
    Accepts:
      - []
      - ['URS...', 'URS...']
      - [['URS...', 'URS...'], ['URS...']]
    Returns:
      - flat list[str]
    """
    if not bucket:
        return []

    flat = []
    for x in bucket:
        if isinstance(x, list):
            flat.extend(x)
        else:
            flat.append(x)
    return flat


def clean_rna_sequence(seq: str) -> str:
    """
    Clean RNA sequence:
    - uppercase
    - convert T -> U
    - keep only A/U/C/G
    """
    seq = seq.upper().replace("T", "U")
    seq = re.sub(r"[^AUCG]", "", seq)
    return seq

def get_rnacentral_sequences(id_list):
    """
    Retrieve sequences from RNAcentral for a list of RNAcentral IDs
    Returns a list of sequences (one per ID)
    """
    sequences = []
    for rc_id in id_list:
        try:
            seq = qrc.rnacentral_to_sequence(rc_id)
            # take first sequence if multiple, clean it
            s = seq[0]
            sequences.append(clean_rna_sequence(s))
        except Exception:
            continue
    return sequences

def get_rnacentral_metadata(id_list):
    """
    Retrieve GO terms and Rfam IDs from RNAcentral for a list of RNAcentral IDs
    Returns two lists: go_terms, rfam_ids
    """
    go_terms = set()
    rfam_ids = set()
    for rc_id in id_list:
        try:
            df_meta = qrc.rnacentral_metadata(rc_id)
            if df_meta is not None and not df_meta.empty:
                go_terms.update(df_meta['go_id'].dropna().unique())
                rfam_ids.update(df_meta['rfam_id'].dropna().unique())
        except Exception:
            continue
    return list(go_terms) if go_terms else pd.NA, list(rfam_ids) if rfam_ids else pd.NA

# ----------------------------
# Apply to DataFrame
# ----------------------------

tqdm.pandas(desc="Fetching RNAcentral sequences & metadata")

def _ensure_list(x):
    """Normalize pandas cell content to a list."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    return [x]  # defensive: single stray value


def enrich_rnacentral(row):
    """
    Merge RNAcentral-derived annotations without overwriting existing ones.
    """
    row = row.copy()

    rc_bucket = row.get("ID_buckets", {}).get("RNAcentral", [])
    rc_ids = normalize_rnacentral_bucket(rc_bucket)

    # ---- sequences ----
    existing_seqs = _ensure_list(row.get("sequences"))
    new_seqs = _ensure_list(get_rnacentral_sequences(rc_ids))

    merged_seqs = list(dict.fromkeys(existing_seqs + new_seqs))
    row["sequences"] = merged_seqs

    # ---- metadata ----
    go_list, rfam_list = get_rnacentral_metadata(rc_ids)

    existing_go = _ensure_list(row.get("GO"))
    existing_rfam = _ensure_list(row.get("Rfam"))

    row["GO"] = list(dict.fromkeys(existing_go + _ensure_list(go_list)))
    row["Rfam"] = list(dict.fromkeys(existing_rfam + _ensure_list(rfam_list)))

    return row


rna_entities_final = rna_entities_trial_stepFour.progress_apply(
    enrich_rnacentral,
    axis=1
)



Fetching RNAcentral sequences & metadata: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 76961/76961 [11:25<00:00, 112.21it/s]


In [37]:
import pandas as pd

# Count rows with at least one sequence
count_with_seq = rna_entities_final['sequences'].apply(lambda x: bool(x)).sum()

def has_valid_entry(x):
    if not isinstance(x, list):
        return False
    # keep only non-missing values
    return any(pd.notna(v) for v in x)

# Count rows with at least one real GO term
count_with_go = rna_entities_final['GO'].apply(has_valid_entry).sum()

# Count rows with at least one real Rfam ID
count_with_rfam = rna_entities_final['Rfam'].apply(has_valid_entry).sum()


# Count rows with at least one GO term
#count_with_go = rna_entities_final['GO'].apply(lambda x: isinstance(x, list) and len(x) > 0).sum() # [<NA>]?? conts as one

# Count rows with at least one Rfam ID
#count_with_rfam = rna_entities_final['Rfam'].apply(lambda x: isinstance(x, list) and len(x) > 0).sum()

total_rows = len(rna_entities_final)

print(f"Total rows: {total_rows}")
print(f"Rows with sequences: {count_with_seq} ({count_with_seq/total_rows:.2%})")
print(f"Rows with GO terms: {count_with_go} ({count_with_go/total_rows:.2%})")
print(f"Rows with Rfam IDs: {count_with_rfam} ({count_with_rfam/total_rows:.2%})")


# ---- total number of sequences (including duplicates) ----
total_sequences = rna_entities_final['sequences'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
).sum()

# ---- total number of unique sequences ----
all_sequences = set()

for seq_list in rna_entities_final['sequences']:
    if isinstance(seq_list, list):
        all_sequences.update(seq_list)

total_unique_sequences = len(all_sequences)

print(f"Total sequences (with redundancy): {total_sequences}")
print(f"Total unique sequences: {total_unique_sequences}")
print(f"Redundancy ratio: {1 - total_unique_sequences/total_sequences:.2%}" if total_sequences > 0 else "No sequences")


# ---- length stratification ----
short_threshold = 4096

total_short = 0
total_long = 0

for seq_list in rna_entities_final['sequences']:
    if isinstance(seq_list, list):
        for seq in seq_list:
            if len(seq) < short_threshold:
                total_short += 1
            else:
                total_long += 1

total_sequences_check = total_short + total_long

print(f"Sequences < {short_threshold} nt: {total_short} ({total_short/total_sequences_check:.2%})")
print(f"Sequences ≥ {short_threshold} nt: {total_long} ({total_long/total_sequences_check:.2%})")


# ---- length stratification ----
short_threshold = 1024

total_short = 0
total_long = 0

for seq_list in rna_entities_final['sequences']:
    if isinstance(seq_list, list):
        for seq in seq_list:
            if len(seq) < short_threshold:
                total_short += 1
            else:
                total_long += 1

total_sequences_check = total_short + total_long

print(f"Sequences < {short_threshold} nt: {total_short} ({total_short/total_sequences_check:.2%})")
print(f"Sequences ≥ {short_threshold} nt: {total_long} ({total_long/total_sequences_check:.2%})")


Total rows: 76961
Rows with sequences: 40730 (52.92%)
Rows with GO terms: 2368 (3.08%)
Rows with Rfam IDs: 2722 (3.54%)
Total sequences (with redundancy): 77834
Total unique sequences: 70553
Redundancy ratio: 9.35%
Sequences < 4096 nt: 74223 (95.36%)
Sequences ≥ 4096 nt: 3611 (4.64%)
Sequences < 1024 nt: 54639 (70.20%)
Sequences ≥ 1024 nt: 23195 (29.80%)


In [38]:
rna_entities_final.to_pickle("data/tmp/rna_entities_final.pkl")

In [40]:
count_false = (rna_entities_final["typecheck"] == False).sum()
print("Rows with typecheck = False:", count_false)

Rows with typecheck = False: 2393


# CircRNA

In [72]:
# Filter rows with Category circRNA, and save txt files with IDs to use in circBase website for batched queries
subset = rna_entities_final[rna_entities_final["Category"].isin(["circRNA"])].copy()

# Compute number of sequences (assumes 'sequences' is a list-like object)
subset["Len_sequences"] = subset["sequences"].apply(lambda x: len(x) if isinstance(x, (list, tuple, set, dict)) else (0 if pd.isna(x) else 1))

# Check condition Len(sequences) >= 1
valid = subset[subset["Len_sequences"] >= 1]
invalid = subset[subset["Len_sequences"] < 1]

print("Total circRNA rows:", len(subset))
print("Rows with Len(sequences) >= 1:", len(valid))
print("Rows with Len(sequences) < 1:", len(invalid))

import os

out_dir = "data/circdata"
os.makedirs(out_dir, exist_ok=True)

def extract_circbase_id(bucket):
    if isinstance(bucket, dict):
        ids = bucket.get("circBase", [])
        if isinstance(ids, list):
            return ids
        elif ids:
            return [ids]
    return []

# Extract circBase IDs
invalid["circ_ids"] = invalid["ID_buckets"].apply(extract_circbase_id)

# Keep rows with at least one circBase ID
invalid = invalid[invalid["circ_ids"].map(len) > 0]

# One row per circBase ID
invalid = invalid.explode("circ_ids")

# Keep only IDs containing 'circ'
invalid = invalid[invalid["circ_ids"].astype(str).str.contains("circ", case=False, na=False)]

# Write one file per species (taxid)
for taxid, group in invalid.groupby("Species"):
    path = os.path.join(out_dir, f"{taxid}.txt")
    with open(path, "w") as f:
        for cid in group["circ_ids"]:
            f.write(f"{cid}\n")

Total circRNA rows: 952
Rows with Len(sequences) >= 1: 27
Rows with Len(sequences) < 1: 925


In [88]:
from pathlib import Path
import re

fasta_dir = Path("data/circdata")

def read_fasta(path):
    seqs = {}
    header = None
    seq_chunks = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header:
                    seq = "".join(seq_chunks).upper().replace("T", "U")
                    seqs[header] = seq
                header = line[1:].split("|")[0]
                seq_chunks = []
            else:
                seq_chunks.append(line)
        if header:
            seq = "".join(seq_chunks).upper().replace("T", "U")
            seqs[header] = seq
    return seqs

# Load sequences per taxid
circbase_sequences = {}
for fa in fasta_dir.glob("circBase_*.fa"):
    taxid = re.search(r"circBase_(\d+)\.fa", fa.name).group(1)
    circbase_sequences[taxid] = read_fasta(fa)


In [89]:
# Map sequences back to dataframe
for idx, row in invalid.iterrows():
    taxid = str(row["Species"])
    cid = row["circ_ids"]
    
    seq = circbase_sequences.get(taxid, {}).get(cid)
    if seq:
        rna_entities_final.at[idx, "sequences"] = [seq]

In [91]:
# Filter rows with Category circRNA, and save txt files with IDs to use in circBase website for batched queries
subset = rna_entities_final[rna_entities_final["Category"].isin(["circRNA"])].copy()

# Compute number of sequences (assumes 'sequences' is a list-like object)
subset["Len_sequences"] = subset["sequences"].apply(lambda x: len(x) if isinstance(x, (list, tuple, set, dict)) else (0 if pd.isna(x) else 1))

# Check condition Len(sequences) >= 1
valid = subset[subset["Len_sequences"] >= 1]
invalid = subset[subset["Len_sequences"] < 1]

print("Total circRNA rows:", len(subset))
print("Rows with Len(sequences) >= 1:", len(valid))
print("Rows with Len(sequences) < 1:", len(invalid))


Total circRNA rows: 952
Rows with Len(sequences) >= 1: 208
Rows with Len(sequences) < 1: 744


In [92]:
compute_stats(rna_entities_final) # new

=== Row-level (entity-level) statistics ===
Ensembl: rows=7496 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=38111 | IDs/row: max=454, mean=1.82, median=1.0
NCBI: rows=39052 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=11672 | IDs/row: max=3, mean=1.00, median=1.0
circBase: rows=257 | IDs/row: max=2, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
tRFdb: rows=928 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=7496, 2=0, 3–10=0, >10=0
RNAcentral: 1=32483, 2=2190, 3–10=2768, >10=670
NCBI: 1=39052, 2=0, 3–10=0, >10=0
miRBase: 1=11650, 2=18, 3–10=4, >10=0
circBase: 1=256, 2=1, 3–10=0, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
tRFdb: 1=928, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 7496 unique IDs
RNAcentral: 62235 unique IDs
NCBI: 38441 unique IDs
miRBase: 11332 unique IDs
circBase: 208 unique IDs
GenBank: 3 unique IDs
tRFdb: 463 unique IDs

=== Sequence availability ===
RNAcentral: 

In [93]:
rna_entities_final.to_pickle("data/tmp/rna_entities_final.pkl")

In [106]:
# remove rows without sequence information (len(sequences list)<1), then save
# Keep only rows with at least one sequence
rna_entities_onlyseq = rna_entities_final[rna_entities_final["sequences"].apply(lambda x: isinstance(x, (list, tuple)) and len(x) > 0)].copy()

# Save to pickle
rna_entities_onlyseq.to_pickle("data/tmp/rna_entities_onlyseq.pkl")
print(len(rna_entities_onlyseq))

40911


<h2>Gene GO term retrieval with mygene TODO</h2>

In [148]:
import pandas as pd
rna_entities_GO_trial_ncRNA = pd.read_pickle("data/tmp/rna_entities_onlyseq.pkl")

In [149]:
len(rna_entities_GO_trial_ncRNA)

40911

In [150]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
print("Start:", datetime.now())
# --- Cache ---
go_cache = {}   # (GeneID, taxid) -> GO dict
raw_cache = {}  # store full API response

# --- Helper to normalize GO fields ---
def normalize_go(block):
    if not block:
        return []
    if isinstance(block, dict):
        block = [block]
    return [
        {
            "id": x.get("id"),
            "term": x.get("term"),
            "evidence": x.get("evidence"),
            "qualifier": x.get("qualifier"),
            "pubmed": x.get("pubmed"),
        }
        for x in block
        if x.get("id") and x.get("term")
    ]

# --- Unified fetch function ---
def fetch_go_batch(pairs, source="NCBI"):
    """
    Fetch GO annotations using GeneID + taxid filtering.
    pairs: list of (GeneID, taxid)
    source: "NCBI" or "Ensembl"
    """
    if not pairs:
        return

    ids = [str(g) for g, _ in pairs]

    # Determine scope for Ensembl
    scope = "ensembl.gene" if source == "Ensembl" else None

    try:
        data = {"ids": ",".join(ids),
                "fields": "go,taxid,symbol,name,type_of_gene,genomic_pos,summary"}
        if scope:
            data["scope"] = scope

        r = requests.post(
            "https://mygene.info/v3/gene",
            data=data,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=60,
        )
        r.raise_for_status()
    except Exception as e:
        print(f"{source} batch failed {ids[:5]}..: {e}")
        return

    resp = r.json()
    tax_map = {str(g): str(t) for g, t in pairs}

    for rec in resp:
        #gid = str(rec.get("entrezgene") or rec.get("_id"))
        gid = str(rec.get("query"))
        taxid_api = str(rec.get("taxid")) if rec.get("taxid") else None

        # enforce strict taxid match
        if gid not in tax_map or taxid_api != tax_map[gid]:
            continue

        go = rec.get("go") or {}
        go_cache[(gid, tax_map[gid])] = {
            "BP": normalize_go(go.get("BP")),
            "MF": normalize_go(go.get("MF")),
            "CC": normalize_go(go.get("CC")),
        }
        raw_cache[(gid, tax_map[gid])] = rec

# --- Prepare subset ---
test_subset = rna_entities_GO_trial_ncRNA #.sample(500, random_state=42).copy()

# --- Build GeneID + taxid pairs ---
def get_geneid_pairs(row):
    pairs = []
    taxid = str(int(row["Species"]))
    bucket = row.get("ID_buckets", {})
    if not isinstance(bucket, dict):
        return pairs

    # NCBI IDs
    ncbi_ids = bucket.get("NCBI", [])
    if isinstance(ncbi_ids, (int, str)):
        ncbi_ids = [ncbi_ids]

    # Ensembl IDs
    ensembl_ids = bucket.get("Ensembl", [])
    if isinstance(ensembl_ids, str):
        ensembl_ids = [ensembl_ids]

    for gid in ncbi_ids:
        if pd.notna(gid):
            pairs.append((str(gid), taxid))
    for gid in ensembl_ids:
        if pd.notna(gid):
            pairs.append((str(gid), taxid))
    return pairs

all_pairs = []
for _, row in test_subset.iterrows():
    all_pairs.extend(get_geneid_pairs(row))

# Remove duplicates
all_pairs = list({(g, t) for g, t in all_pairs})

# --- Split by source for API scope ---
ncbi_pairs = [p for p in all_pairs if p[0].isdigit()]
ensembl_pairs = [p for p in all_pairs if not p[0].isdigit()]

# --- batching ---
batch_size = 1000
def batch_list(lst):
    return [lst[i:i+batch_size] for i in range(0, len(lst), batch_size)]

ncbi_batches = batch_list(ncbi_pairs)
ensembl_batches = batch_list(ensembl_pairs)

# --- Parallel fetch ---
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(fetch_go_batch, batch, "NCBI") for batch in ncbi_batches]
    futures += [executor.submit(fetch_go_batch, batch, "Ensembl") for batch in ensembl_batches]
    for f in as_completed(futures):
        f.result()

# --- Map GO back to DataFrame ---
def map_go(row):
    taxid = str(int(row["Species"]))
    bucket = row.get("ID_buckets", {})
    result = {"BP": [], "MF": [], "CC": []}
    for source in ["NCBI", "Ensembl"]:
        ids = bucket.get(source, [])
        if isinstance(ids, (int, str)):
            ids = [ids]
        for gid in ids:
            key = (str(gid), taxid)
            if key in go_cache:
                for k in ["BP", "MF", "CC"]:
                    result[k].extend(go_cache[key][k])
    # deduplicate
    for k in result:
        seen = set()
        result[k] = [x for x in result[k] if x["id"] not in seen and not seen.add(x["id"])]
    return result

test_subset["GO_raw"] = test_subset.apply(map_go, axis=1)
test_subset["GO_BP_count"] = test_subset["GO_raw"].apply(lambda d: len(d["BP"]))
test_subset["GO_MF_count"] = test_subset["GO_raw"].apply(lambda d: len(d["MF"]))
test_subset["GO_CC_count"] = test_subset["GO_raw"].apply(lambda d: len(d["CC"]))

# --- Keep raw payload ---
def map_raw(row):
    taxid = str(int(row["Species"]))
    bucket = row.get("ID_buckets", {})
    for source in ["NCBI", "Ensembl"]:
        ids = bucket.get(source, [])
        if isinstance(ids, (int, str)):
            ids = [ids]
        for gid in ids:
            key = (str(gid), taxid)
            if key in raw_cache:
                return raw_cache[key]
    return None

test_subset["mygene_raw"] = test_subset.apply(map_raw, axis=1)

print("Done:", datetime.now())

Start: 2026-03-11 15:29:08.359737
Done: 2026-03-11 15:29:34.461811


In [151]:
len(test_subset)

40911

In [153]:
test_subset.to_pickle("data/nc_rna_entities_processed.pkl")